# N074 · 树的构建与序列化

**目标：** 在充分的编码约束下恢复树结构。

**先修：** N019, N069。本章自包含，依次运行即可；建议先读推导，再预测输出。

**知识点：** 前中序划分；索引映射；空节点标记；可逆编码；重复值歧义。

**配套讲解来源：** [同名逐章意见](../../comment/074_tree_construction_serialization.md)。

本章阅读顺序：**先明白问题 → 补基础 → 手算 → 看核心片段 → 分段运行 → 核对测试**。需要复制使用时，跳到末尾“完整代码”，它包含全部依赖、实现和测试。

## 1. 通俗解释：输入什么，要得到什么

树是长在内存里的对象，你没法直接把它存进文件或在网络上传输。本章解决两个方向的"翻译"问题：第一，**重建**——给你一棵树的前序遍历和中序遍历两个序列，把树原样拼回来（`build_tree`）；第二，**序列化/反序列化**——把树压成一个字符串（`serialize`），再从字符串无损地还原（`deserialize`）。

举一个具体到数字的例子。输入前序 `[3,9,20,15,7]`、中序 `[9,3,15,20,7]`，`build_tree` 要输出这棵树：

```
        3
       / \
      9   20
          / \
        15   7
```

为什么是它？前序的第一个值 3 一定是根；在中序里找到 3，它左边只有 9，所以 9 是整个左子树，右边 `[15,20,7]` 是右子树；对右子树递归地做同样的事：前序里下一个是 20（右子树的根），中序里 20 左边是 15、右边是 7，格局就定了。再举序列化的例子：上面那棵树 `serialize` 输出字符串 `'3,9,#,#,20,15,#,#,7,#,#'`，其中 `#` 表示空孩子；把这段字符串喂给 `deserialize`，能一字不差地还原出原树。为什么必须在字符串里写 `#`？因为只写值的话，`'3,9'` 既可能是"3 的左孩子是 9"也可能是"右孩子是 9"，形状信息丢了——这是练习 1 的主题。

## 2. 基础知识：先读懂这些词

- **前序遍历（preorder）**：按"自己、左子树、右子树"的顺序访问。它有个关键性质：第一个值一定是根，且子树里的值在前序里是连续排列的。
- **中序遍历（inorder）**：按"左子树、自己、右子树"的顺序访问。关键性质：根的位置就是分界线，左边全是左子树的值，右边全是右子树的值。前序告诉你"谁先当根"，中序告诉你"左右各有多少人"，两个信息合起来才能唯一确定树。
- **重复值歧义**：重建算法要求所有节点值互不相同。如果值有重复，中序里可能找到好几个"根的值"，切分位置不再唯一，同一对前序+中序可能对应好几棵不同的树，重建就没有确定答案了。
- **空节点标记（#）**：序列化时把 `None` 指针也显式写成 `#`。空指针看似"没信息"，其实它正是"这个节点没有左孩子/右孩子"的形状信息，缺了它就没法还原。
- **可逆编码（往返一致）**：一段编码是可逆的，指的是"编码再解码再编码"得到一模一样的字符串。本章测试反复用 `serialize(deserialize(text))==text` 来验证这一点。
- **槽（slot）结构**：反序列化时的一种记账方式：每个还没挂满孩子的非空节点提供两个空位（先左后右），消费掉一个空位后，如果挂上的是非空节点，它自己又带来两个新空位。只要按这个规则走，编码与树一一对应；结束时空位必须恰好用完，多出来或不够都说明编码是坏的。
- **迭代代替递归**：重建和序列化都用显式栈写成迭代。测试里专门造了一棵 1500 层的深链——超过 Python 默认约 1000 层的递归上限——迭代版本毫无压力，递归版本会直接崩。

## 3. 思路推导：从小例子开始

### 1. `build_tree`：栈版"前序+中序"重建

- **Step 1**：前序第一个值是根，建根节点压栈。栈里存的是"还没有匹配到中序位置"的祖先链，`index` 指向中序里下一个等待被匹配的值（从 0 开始）。
- **Step 2**：逐个处理前序剩下的值，每来一个值只可能是两种身份之一：要么是栈顶节点的左孩子（因为栈顶的左子树还没轮完），要么是"某个祖先的左子树已经结束"后接上去的右孩子。
- **Step 3**：怎么判断是哪种身份？拿栈顶的值和中序 `index` 位置的值比：不相等，说明栈顶的左子树还没走完，新值就是它的左孩子；相等，说明栈顶这个节点在中序里"到点了"（它自己该出场了，左子树结束），把它弹出、`index` 前进，可能连着好几个祖先都到点，就一直弹，最后弹出的那个祖先的右孩子就是新值。
- **Step 4**：手算前序 `[3,9,20,15,7]`、中序 `[9,3,15,20,7]`：
  - 根 3 进栈，栈 [3]，index=0（指着 9）。
  - 来 9：栈顶 3 ≠ 中序 9 → 9 是 3 的左孩子。栈 [3,9]。
  - 来 20：栈顶 9 == 中序 9 → 弹出 9，index=1；栈顶 3 == 中序 3 → 弹出 3，index=2；栈空停。最后弹出的 3 的右孩子是 20。栈 [20]。
  - 来 15：栈顶 20 ≠ 中序 15 → 15 是 20 的左孩子。栈 [20,15]。
  - 来 7：栈顶 15 == 中序 15 → 弹出，index=3；栈顶 20 == 中序 20 → 弹出，index=4；栈空停。20 的右孩子是 7。栈 [7]。
  - 收尾：栈里的 7 == 中序最后一位 7，弹出，index=5 正好走完。栈空、中序用尽，重建成功，正是第一节那棵树。
- **Step 5**：收尾后如果栈里还有剩、或中序没消费完，说明两个序列根本不是同一棵树产生的，抛异常拒绝服务。

### 2. `serialize` / `deserialize`：带 # 的前序文本

- **Step 6**：序列化就是"把空指针也吐出来的前序遍历"：访问一个节点就写下它的值（`None` 写 `#`），然后先右后左地把孩子压栈（栈后进先出，所以实际处理顺序是先左后右，保证是前序）。拿 “运行示例”部分 的小树 `[1,-2,3,None,4]` 手算：前序依次是 1, -2, 4, 空空, 空, 3, 空空，得到 `'1,-2,4,#,#,#,3,#,#'`。“运行示例”部分 的表格把每个 token 连同下标列了出来，并验证了"解码再编码"得到同样的字符串。
- **Step 7**：反序列化就是对偶过程：第一个 token 是根；此后每个 token 消费"当前最靠左的空位"——即栈顶节点还没挂的左槽或右槽。手算 `'3,9,#,#,20,15,#,#,7,#,#'`：3 是根；9 填 3 的左槽；# 填 9 的左槽；# 填 9 的右槽（9 挂满了退场）；20 填 3 的右槽（3 退场）；15 填 20 的左槽；两个 # 挂满 15 退场；7 填 20 的右槽（20 退场）；两个 # 挂满 7 退场。栈空收工，树还原。
- **Step 8**：错误检测：token 用完栈里还有节点等着挂孩子，说明"树被截断"；栈空了还有 token 剩下，说明"编码有多余尾巴"；两种都抛 `ValueError`。空字符串 `''` 切出来的空 token 传给 `int()` 也会抛 `ValueError`，坏输入统一报错。

## 4. 核心代码：build_tree

先关注决定答案的状态与更新操作。这是完整实现的对应片段，不是另一套解法；导入、辅助结构及其余接口在下一节补全。

```python
def build_tree(preorder, inorder):
    if len(preorder) != len(inorder) or len(set(preorder)) != len(preorder) or set(preorder) != set(inorder):
        raise ValueError('matching traversals with distinct keys required')
    if not preorder:
        return None
    root = TreeNode(preorder[0])
    stack = [root]
    index = 0
    for value in preorder[1:]:
        node = TreeNode(value)
        if stack[-1].val != inorder[index]:
            stack[-1].left = node
        else:
            parent = None
            while stack and index < len(inorder) and (stack[-1].val == inorder[index]):
                parent = stack.pop()
                index += 1
            parent.right = node
        stack.append(node)
    while stack and index < len(inorder) and (stack[-1].val == inorder[index]):
        stack.pop()
        index += 1
    if stack or index != len(inorder):
        raise ValueError('inconsistent traversals')
    return root
```

### 逐段读懂代码

`TreeNode`、`tree_from_level` 与前几章一致，跳过。“分段实现”部分 有三个新函数。

### 1. 前中序迭代重建

（对应程序见下方分段实现与完整代码。）

逐行看：

- 第一行的三连检查是"合同审查"：两个序列长度必须相同（`len(...)!=len(...)`）、前序里不能有重复值（`len(set(preorder))!=len(preorder)` 说明去重后变短了，即有重复）、两个序列的值的集合必须一致（`set(preorder)!=set(inorder)`）。任何一条不满足就抛 `ValueError`，因为重建的前提被破坏了。
- `if not preorder: return None` 处理空树。
- 主循环逐个消费前序的值。`stack[-1].val!=inorder[index]` 是"身份判定"：栈顶的左子树还没轮完，新值挂在栈顶左边；`else` 分支处理"到点"：`while` 循环把所有在中序里排到出场顺序的祖先连续弹出，`index` 同步前进，最后弹出的 `parent` 接住新值当右孩子。`parent=None` 只是初始化，`while` 至少会弹一次（因为进入 else 的条件就是栈顶匹配）。
- 每个新节点最后都 `stack.append(node)` 进栈，成为下一段的"待匹配祖先"。
- 循环后的 `while` 是收尾清算：最右侧那串祖先也要按中序顺序退场。
- 最后一行是终审：栈非空（有节点从未被中序匹配）或 `index` 没走到头（中序有值没消费）都说明输入自相矛盾，抛 `'inconsistent traversals'`。

### 2. 序列化：空指针显式写盘

```python
def serialize(root):
    tokens = []
    stack = [root]
    while stack:
        node = stack.pop()
        if node is None:
            tokens.append('#')
        else:
            tokens.append(str(node.val))
            stack.append(node.right)
            stack.append(node.left)
    return ','.join(tokens)
```

栈弹出谁就处理谁：`None` 写 `'#'`；正常节点把值转成字符串记下，然后**先压右孩子再压左孩子**——栈是后进先出的，先压右后压左，下次先弹左，输出顺序才是"自己、左、右"的前序。最后用逗号拼成一个字符串。

### 3. 反序列化：按槽消费 token

```python
def deserialize(data):
    tokens = data.split(',')

    def node_from(token):
        return None if token == '#' else TreeNode(int(token))
    root = node_from(tokens[0])
    stack = [[root, 0]] if root else []
    for token in tokens[1:]:
        if not stack:
            raise ValueError('extra tokens')
        child = node_from(token)
        parent, side = stack[-1]
        if side == 0:
            parent.left = child
            stack[-1][1] = 1
        else:
            parent.right = child
            stack.pop()
        if child is not None:
            stack.append([child, 0])
    if stack:
        raise ValueError('truncated tree')
    return root
```

- `node_from` 是个一行函数：`#` 变 `None`，其他 token 转成整数造节点。
- 栈里每项是 `[节点, 下一个待填的槽]`：`0` 表示左槽还没填，`1` 表示左槽已填、等右槽。
- 第一个 token 造根；`stack=[[root,0]] if root else []` 是紧凑写法——根是 `#`（空树）时给空栈，后面循环自然全部跳过。
- 每来一个 token：栈空说明没地方挂了，抛 `'extra tokens'`；否则看栈顶槽位，`side==0` 就挂左边并把槽位翻成 1，`side` 已是 1 就挂右边并让这个父亲整体退场（`stack.pop()`）。挂上的孩子如果不是 `None`，它自己带着两个新槽进栈。
- 结束时栈非空，说明还有节点欠着孩子没填——树被截断了，抛 `'truncated tree'`。

## 5. 分段实现：按顺序运行

**本章接口：** `build_tree(preorder, inorder)`、`serialize(root)`、`deserialize(data)`

### TreeNode

In [1]:
class TreeNode:

    def __init__(self, val=0, left=None, right=None):
        self.val, self.left, self.right = (val, left, right)

### tree_from_level

In [2]:
def tree_from_level(values):
    """Construct a finite binary tree from level order with None placeholders."""
    from collections import deque
    if not values or values[0] is None:
        return None
    root = TreeNode(values[0])
    queue = deque([root])
    i = 1
    while queue and i < len(values):
        node = queue.popleft()
        if values[i] is not None:
            node.left = TreeNode(values[i])
            queue.append(node.left)
        i += 1
        if i < len(values):
            if values[i] is not None:
                node.right = TreeNode(values[i])
                queue.append(node.right)
            i += 1
    return root

### build_tree

In [3]:
def build_tree(preorder, inorder):
    if len(preorder) != len(inorder) or len(set(preorder)) != len(preorder) or set(preorder) != set(inorder):
        raise ValueError('matching traversals with distinct keys required')
    if not preorder:
        return None
    root = TreeNode(preorder[0])
    stack = [root]
    index = 0
    for value in preorder[1:]:
        node = TreeNode(value)
        if stack[-1].val != inorder[index]:
            stack[-1].left = node
        else:
            parent = None
            while stack and index < len(inorder) and (stack[-1].val == inorder[index]):
                parent = stack.pop()
                index += 1
            parent.right = node
        stack.append(node)
    while stack and index < len(inorder) and (stack[-1].val == inorder[index]):
        stack.pop()
        index += 1
    if stack or index != len(inorder):
        raise ValueError('inconsistent traversals')
    return root

### serialize

In [4]:
def serialize(root):
    tokens = []
    stack = [root]
    while stack:
        node = stack.pop()
        if node is None:
            tokens.append('#')
        else:
            tokens.append(str(node.val))
            stack.append(node.right)
            stack.append(node.left)
    return ','.join(tokens)

### deserialize

In [5]:
def deserialize(data):
    tokens = data.split(',')

    def node_from(token):
        return None if token == '#' else TreeNode(int(token))
    root = node_from(tokens[0])
    stack = [[root, 0]] if root else []
    for token in tokens[1:]:
        if not stack:
            raise ValueError('extra tokens')
        child = node_from(token)
        parent, side = stack[-1]
        if side == 0:
            parent.left = child
            stack[-1][1] = 1
        else:
            parent.right = child
            stack.pop()
        if child is not None:
            stack.append([child, 0])
    if stack:
        raise ValueError('truncated tree')
    return root

## 6. 运行示例与状态可视化

先预测结果再运行。手算表帮助解释过程；下面的表由当前实现计算。若表只展示输入输出，它并不代替内部状态轨迹。

In [6]:
from IPython.display import display, HTML
from html import escape

def show_table(headers, rows):
    """Render a small, explicit state trace; no network or hidden algorithm code."""
    head = ''.join('<th>' + escape(str(x)) + '</th>' for x in headers)
    body = ''.join('<tr>' + ''.join('<td>' + escape(str(x)) + '</td>' for x in row) + '</tr>' for row in rows)
    display(HTML('<table><thead><tr>' + head + '</tr></thead><tbody>' + body + '</tbody></table>'))

r=tree_from_level([1,-2,3,None,4]); text=serialize(r)
show_table(['token索引','前序token'],list(enumerate(text.split(','))))
show_table(['原编码','往返编码'],[(text,serialize(deserialize(text)))])

token索引,前序token
0,1
1,-2
2,#
3,4
4,#
5,#
6,3
7,#
8,#


原编码,往返编码
"1,-2,#,4,#,#,3,#,#","1,-2,#,4,#,#,3,#,#"


## 7. 正确性、适用条件与复杂度

前序确定新节点出现次序，中序匹配确定何时结束一个祖先的左侧并转向右侧。序列化中每个非空节点消费一个槽、产生两个子槽，#消费一个槽，因此完整槽结构唯一解码。

**代价：** O(n)时间与O(n)辅助空间；文本长度还依赖整数位数。迭代实现可处理深链，不受递归限制。

### 展开理解

**重建为什么对？** 前序规定了新节点出场的先后次序；中序规定了"每个节点什么时候轮完自己的左子树"。栈里存的恰好是"从根到当前节点这条右侧祖先链上、左子树还没轮完的节点"，而 `index` 指向中序里下一个该出场的值。当栈顶的值和中序 `index` 处相等，就意味着这个节点在中序里出场了——它的左子树到此为止——所以把它弹出、让 index 前进，弹到不相等为止，最后弹出的那个就是"左子树刚结束、正缺右孩子"的祖先。这两条信息互相咬合，每一步的挂法都是唯一确定的。

**序列化为什么可逆？** 数一数"槽"就够了：每个非空节点消费一个槽、同时贡献两个新槽；每个 `#` 消费一个槽、不贡献新槽。按前序顺序消费，编码和"树+空指针的形状"一一对应，所以解码过程每一步同样唯一——没有选择余地，就没有歧义。结束时槽不多不少，正是一个自洽的完整编码。

**复杂度**：三个函数都是每个节点（含空指针）进出栈常数次，时间 O(n)；栈和 token 列表都是 O(n) 辅助空间；序列化文本的长度除了节点数还取决于整数的位数（数越大字符串越长，比如 12 位数比 1 位数多占几个字符）。n=10⁵ 的树，序列化也就是百万字符量级的拼接，很快。迭代写法不受递归深度限制，测试里那条 1500 层的右链（前序=中序=[0,1,...,1499]）就是专门来证明这一点的。

## 8. 单元测试：每个用例在检查什么

- `r=build_tree([3,9,20,15,7],[9,3,15,20,7]); assert serialize(r)=='3,9,#,#,20,15,#,#,7,#,#'`：重建的标准用例（LeetCode 105 的样例树），验证方式很巧——不直接比对树对象，而是把重建结果序列化成文本比对，文本相同即树相同。
- `for values in [[],[0],[-1,2,3,None,4],[1,None,2,None,3]]` 的往返测试：四棵形态各异的树（空树、单节点、带负数、左侧一条链）逐一验证"序列化→反序列化→再序列化"得到同样文本。空树测的是 `#` 单独成串的边界；负数 `-1` 测 `int(token)` 对负号的兼容。
- `assert deserialize('#') is None`：只有一个空标记的字符串就是空树的标准编码。
- `for bad in ['1,#,#,#','1,#,#,2','']`：三个坏编码。`'1,#,#,#'` 和 `'1,#,#,2'` 在树挂满后还有多余 token，应抛 `'extra tokens'`；`''` 切出空 token，`int('')` 抛 `ValueError`。断言它们"必须失败"，防止实现悄悄容忍垃圾输入。
- `build_tree([1,2,3],[3,1,2])`：长度、去重、集合三项审查都过关，但两个序列根本不是同一棵树的前序和中序，必须靠收尾审查抓出来抛 `'inconsistent traversals'`。它测的是"合同审查查不出的深层矛盾"。
- 1500 层右链的往返测试：一石二鸟，既验证大输入正确性，又证明迭代实现不受 Python 递归深度限制。

In [7]:
r = build_tree([3, 9, 20, 15, 7], [9, 3, 15, 20, 7])

assert serialize(r) == '3,9,#,#,20,15,#,#,7,#,#'

for values in [[], [0], [-1, 2, 3, None, 4], [1, None, 2, None, 3]]:
    text = serialize(tree_from_level(values))
    assert serialize(deserialize(text)) == text

assert deserialize('#') is None

for bad in ['1,#', '#,#', '1,#,#,2', '']:
    try:
        deserialize(bad)
    except ValueError:
        pass
    else:
        raise AssertionError('malformed encoding must fail')

try:
    build_tree([1, 2, 3], [3, 1, 2])
except ValueError:
    pass
else:
    raise AssertionError('inconsistent traversals must fail')

a = list(range(1500))

r = build_tree(a, a)

assert serialize(deserialize(serialize(r))) == serialize(r)

print('N074: 所有本章断言通过')

N074: 所有本章断言通过


## 9. 练习：先尝试，再看提示

**练习 1：** 说明缺少空标记为何可能歧义。

**练习 2：** 设计非法编码的明确错误。

<details>
<summary>展开思路提示（不是完整答案）</summary>

- **练习 1（说明缺空标记为何歧义）**：提示——只用值做前序编码，手写 `'1,2'` 的两种还原：1 的左孩子是 2，和 1 的右孩子是 2，两棵树不同但编码相同，这本身就证明了歧义。再想一个更隐蔽的：`'1,2,3'` 能画出几棵不同的树？（至少左链、右链、"1 左 2、2 右 3"三种。）关键句是：形状信息不在值里，在"哪里缺孩子"里，`#` 正是把缺孩子这件事写进了编码。
- **练习 2（为非法编码设计明确错误）**：提示——现在 `''` 这种输入的错误来自 `int('')` 的副作用，报错信息看不出"是编码坏了还是数字格式坏了"。想想非法编码能分几类：token 用不完（尾巴多余）、槽填不满（中途截断）、token 内容既不是 `#` 也不是整数。给每类设计一个专属报错消息（例如 `'empty input'`、`'bad token: xxx'`），并在 `deserialize` 开头把空字符串单独拦下。先写三个坏输入的手算预期，再改代码让它们各报各的错。

</details>

## 10. 完整代码：可独立运行

下面这一格包含**全部导入、全部接口和验证用例**，不依赖前面的单元格。可以重启内核后只运行这一格，也可以复制到 `.py` 文件后运行。在 Notebook 中 Run All 时会再验证一次，这是为独立复制使用保留的完整版本。

In [8]:
# 完整实现：本单元格不依赖其他单元格。
class TreeNode:

    def __init__(self, val=0, left=None, right=None):
        self.val, self.left, self.right = (val, left, right)

def tree_from_level(values):
    """Construct a finite binary tree from level order with None placeholders."""
    from collections import deque
    if not values or values[0] is None:
        return None
    root = TreeNode(values[0])
    queue = deque([root])
    i = 1
    while queue and i < len(values):
        node = queue.popleft()
        if values[i] is not None:
            node.left = TreeNode(values[i])
            queue.append(node.left)
        i += 1
        if i < len(values):
            if values[i] is not None:
                node.right = TreeNode(values[i])
                queue.append(node.right)
            i += 1
    return root

def build_tree(preorder, inorder):
    if len(preorder) != len(inorder) or len(set(preorder)) != len(preorder) or set(preorder) != set(inorder):
        raise ValueError('matching traversals with distinct keys required')
    if not preorder:
        return None
    root = TreeNode(preorder[0])
    stack = [root]
    index = 0
    for value in preorder[1:]:
        node = TreeNode(value)
        if stack[-1].val != inorder[index]:
            stack[-1].left = node
        else:
            parent = None
            while stack and index < len(inorder) and (stack[-1].val == inorder[index]):
                parent = stack.pop()
                index += 1
            parent.right = node
        stack.append(node)
    while stack and index < len(inorder) and (stack[-1].val == inorder[index]):
        stack.pop()
        index += 1
    if stack or index != len(inorder):
        raise ValueError('inconsistent traversals')
    return root

def serialize(root):
    tokens = []
    stack = [root]
    while stack:
        node = stack.pop()
        if node is None:
            tokens.append('#')
        else:
            tokens.append(str(node.val))
            stack.append(node.right)
            stack.append(node.left)
    return ','.join(tokens)

def deserialize(data):
    tokens = data.split(',')

    def node_from(token):
        return None if token == '#' else TreeNode(int(token))
    root = node_from(tokens[0])
    stack = [[root, 0]] if root else []
    for token in tokens[1:]:
        if not stack:
            raise ValueError('extra tokens')
        child = node_from(token)
        parent, side = stack[-1]
        if side == 0:
            parent.left = child
            stack[-1][1] = 1
        else:
            parent.right = child
            stack.pop()
        if child is not None:
            stack.append([child, 0])
    if stack:
        raise ValueError('truncated tree')
    return root

# 示例与回归测试
r = build_tree([3, 9, 20, 15, 7], [9, 3, 15, 20, 7])

assert serialize(r) == '3,9,#,#,20,15,#,#,7,#,#'

for values in [[], [0], [-1, 2, 3, None, 4], [1, None, 2, None, 3]]:
    text = serialize(tree_from_level(values))
    assert serialize(deserialize(text)) == text

assert deserialize('#') is None

for bad in ['1,#', '#,#', '1,#,#,2', '']:
    try:
        deserialize(bad)
    except ValueError:
        pass
    else:
        raise AssertionError('malformed encoding must fail')

try:
    build_tree([1, 2, 3], [3, 1, 2])
except ValueError:
    pass
else:
    raise AssertionError('inconsistent traversals must fail')

a = list(range(1500))

r = build_tree(a, a)

assert serialize(deserialize(serialize(r))) == serialize(r)

print('N074: 所有本章断言通过')

N074: 所有本章断言通过


## 11. 代表题与迁移

- **105. Construct Binary Tree from Preorder and Inorder Traversal**：前序+中序重建，直接对应 `build_tree`，本章的栈解法是不用递归、不用哈希索引的另一种经典写法。
- **106. Construct Binary Tree from Inorder and Postorder Traversal**：中序+后序重建，思路镜像对称（后序从右往左看，新值先挂右孩子），练的是把本章"栈顶匹配中序"的技巧反着用。
- **297. Serialize and Deserialize Binary Tree**：设计编解码器，直接对应 `serialize`/`deserialize`，本章"前序+空标记"是最基础的可行方案之一（官方还有层序、括号嵌套等变体）。

**题面入口：** 

题目用于知识映射；本章完整代码遵循教学接口，不等于每道题的完整平台提交。

## 12. 掌握检查

合上代码，说明输入输出、核心变量的含义、每次更新为什么成立，以及一个边界情况。再完成一道练习，不能只凭“运行通过”判断掌握。

**继续学习：** [课程索引](../../COURSE_INDEX.md)。